# AI Personal Trainer on IBM watsonx.ai

> **Not medical advice.** Educational project; check with a qualified professional before starting a new program.

Photos of the gym equipment you have → a watsonx.ai vision model lists the equipment → a watsonx.ai
text model writes a weekly plan that uses only that equipment → the notebook checks the plan against
the equipment list and exports an HTML report.

**Setup:** `WATSONX_APIKEY`, `WATSONX_PROJECT_ID` and `WATSONX_URL` in the repo's `.env`, and one or more
equipment photos in `data/gym/` (jpg, png or webp). Models can be changed with `WATSONX_TEXT_MODEL` and
`WATSONX_VISION_MODEL`; IBM retires models regularly, so check what your region offers.

In [ ]:
# Cell 1 — Settings and watsonx.ai connection
import os, re, json, base64
from datetime import date
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

load_dotenv(find_dotenv(usecwd=True))
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

missing = [k for k in ("WATSONX_APIKEY", "WATSONX_PROJECT_ID", "WATSONX_URL") if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Set {', '.join(missing)} in the repo's .env (see .env.example)")

TEXT_MODEL = os.getenv("WATSONX_TEXT_MODEL", "meta-llama/llama-3-3-70b-instruct")
VISION_MODEL = os.getenv("WATSONX_VISION_MODEL", "meta-llama/llama-4-maverick-17b-128e-instruct-fp8")
credentials = Credentials(url=os.environ["WATSONX_URL"], api_key=os.environ["WATSONX_APIKEY"])

def watsonx_model(model_id, **params):
    return ModelInference(model_id=model_id, credentials=credentials,
                          project_id=os.environ["WATSONX_PROJECT_ID"], params=params)

def reply_text(resp) -> str:
    return (resp.get("choices", [{}])[0].get("message", {}).get("content") or "").strip()

print(f"Region: {os.environ['WATSONX_URL']}\nText model: {TEXT_MODEL}\nVision model: {VISION_MODEL}")

In [ ]:
# Cell 2 — Load the equipment photos
GYM_DIR = PROJECT_DIR / "data" / "gym"
paths = sorted(p for p in GYM_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
if not paths:
    raise FileNotFoundError(f"Put one or more photos of gym equipment in {GYM_DIR}")

MIME = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}
images = [(p.name, MIME[p.suffix.lower()], base64.b64encode(p.read_bytes()).decode()) for p in paths]
print(f"Loaded {len(images)} photo(s):", [name for name, _, _ in images])

In [ ]:
# Cell 3 — Vision model: what equipment is in each photo?
VISION_PROMPT = (
    "You are a careful personal-trainer assistant. List the gym equipment clearly visible in this photo. "
    "Don't guess at things you can't see. Reply with JSON only, in this form: "
    '{"equipment": ["adjustable dumbbells", "flat bench"], "category": "strength_training", "workout_type": "upper_body"}. '
    "category is one of strength_training, cardio, flexibility, other. "
    "workout_type is one of upper_body, lower_body, full_body, core, cardio."
)

def parse_json(text: str) -> dict:
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        raise ValueError(f"No JSON in the model's reply:\n{text}")
    return json.loads(match.group(0))

vision = watsonx_model(VISION_MODEL, max_tokens=300, temperature=0)
gym_equipment = []
for name, mime, b64 in images:
    resp = vision.chat(messages=[{"role": "user", "content": [
        {"type": "text", "text": VISION_PROMPT},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ]}])
    raw = reply_text(resp)
    found = parse_json(raw)
    gym_equipment.append({"filename": name, "description": ", ".join(found.get("equipment", [])),
                          "category": found.get("category", ""), "workout_type": found.get("workout_type", "")})
    print(f"{name}: {raw}")

In [ ]:
# Cell 4 — Detected equipment
df_equipment = pd.DataFrame(gym_equipment)
df_equipment

In [ ]:
# -----------------------------------------------------------
# Cell 7: Define the USER PROFILE (who the plan is for)
# -----------------------------------------------------------
# The workout plan depends heavily on:
#   - age
#   - fitness level
#   - training experience
#   - personal goals
#   - available days per week
#   - injuries or limitations
#   - exercise preferences
#
# This block allows students to MODIFY these inputs and see how
# the planner model changes the weekly workout plan.
#
# This is an excellent place for experimentation:
#   - Change goals ("weight loss", "hypertrophy", "flexibility")
#   - Change availability (2 vs 5 days per week)
#   - Add injuries ("knee pain", "no overhead lifting")
#   - Try different preference sets
#
# The planner model adapts the plan to these inputs.
# -----------------------------------------------------------

user_profile = {
    "name": "Gurveer",                         # User's name (only for display)
    "age": 28,                              # Age can influence exercise intensity
    "gender": "Male",                # Optional metadata

    # Beginner / intermediate / advanced
    # This will affect volume, exercise difficulty, and selection.
    "fitness_level": "intermediate",

    # Experience with lifting influences safety + complexity of movements.
    "experience_with_strength_training": "moderate",  # none / low / moderate / high

    # A list of goals; the model will integrate these into the plan structure.
    "primary_goals": [
        "build strength",
        "improve general fitness"
    ],

    # Number of workout days the plan should cover.
    "days_per_week_available": 4,

    # A realistic session length helps ensure the plan isn’t too long.
    "session_length_minutes": 50,

    # List any injuries—model avoids unsafe exercises when possible.
    "injuries_or_limitations": [],

    # Personal likes and dislikes guide exercise selection.
    "exercise_preferences": {
        "likes": ["free weights"],
        "dislikes": ["high impact jumping"]
    }
}

# -----------------------------------------------------------
# Convert both the user profile and equipment list into JSON
# -----------------------------------------------------------
# The planner model receives these values AS TEXT in the prompt.
# JSON formatting ensures:
#   - clean structure
#   - predictable formatting
#   - easier parsing for LLMs
# -----------------------------------------------------------

user_profile_json = json.dumps(user_profile, indent=2)
equipment_json = json.dumps(gym_equipment, indent=2)

# (Optional for debugging)
print("User profile (JSON):")
print(user_profile_json)

print("\nEquipment list (JSON):")
print(equipment_json)

In [ ]:
# Cell 6 — Text model: weekly plan using only the detected equipment
COLUMNS = ["Day", "Focus", "Exercise", "Equipment", "Sets", "RepsOrTime", "TargetMuscle",
           "Intensity", "Warmup", "Cooldown", "DayNotes"]
system = (
    "You are an experienced, safety-conscious personal trainer. Use only the equipment listed, or bodyweight. "
    "Respect injuries and dislikes. Reply with a pipe-delimited table only: no explanations, no code fences, "
    "and no '|' inside any value. The first row is the header: " + " | ".join(COLUMNS)
)
user = f"USER PROFILE:\n{user_profile_json}\n\nAVAILABLE EQUIPMENT:\n{equipment_json}"

planner = watsonx_model(TEXT_MODEL, max_tokens=1500, temperature=0.2)
plan_text = reply_text(planner.chat(messages=[{"role": "system", "content": system},
                                              {"role": "user", "content": user}]))
if "|" not in plan_text:
    raise ValueError(f"The planner didn't return a table:\n{plan_text}")
print(plan_text)

In [ ]:
# -----------------------------------------------------------
# Cell 10: Parse the workout plan → Create DataFrame → Build HTML report
# -----------------------------------------------------------
# This is the final “assembly” cell. It does FOUR major things:
#
# 1.  SAFELY parse the raw planner output (plan_text)
#     The model returns plain text, so we:
#        - split into lines
#        - detect the header row
#        - detect the exercise rows
#        - handle formatting mistakes
#
# 2.  Convert the structured rows into a Pandas DataFrame
#     This becomes df_plan, which is our final “clean table”.
#
# 3.  Build a readable HTML report with:
#        - a header block (user profile summary)
#        - a section for each day with an exercise table
#
# 4.  Save the final HTML file ("workout_plan.html") to disk
#     You can open it in any browser → Print → Save as PDF.
# -----------------------------------------------------------

import pandas as pd
import os

# -----------------------------
# 1) Parse planner output (plan_text) into df_plan
# -----------------------------

# Safety check: make sure the planner produced real text
if not isinstance(plan_text, str) or not plan_text.strip():
    raise ValueError(
        "plan_text is empty or not a string.\n"
        "Make sure the planner_model cell ran correctly and returned text."
    )

# Split output into individual non-empty lines
lines = [l.strip() for l in plan_text.splitlines() if l.strip()]

# Keep only lines that contain the pipe character '|'
# This ensures we ignore any accidental text the model might add.
table_lines = [l for l in lines if "|" in l]

if not table_lines:
    raise ValueError(
        "No table-like lines found in the plan_text.\n"
        "Check the raw planner output above."
    )

# The first line is the header with column names
header_line = table_lines[0]

# Everything after the header is a data row
data_lines = table_lines[1:]

# Expected column names (exact order)
expected_cols = [
    "Day", "Focus", "Exercise", "Equipment", "Sets",
    "RepsOrTime", "TargetMuscle", "Intensity",
    "Warmup", "Cooldown", "DayNotes"
]

# Show the parsed header so students can understand what’s going on
header_parts = [p.strip() for p in header_line.split("|")]
print("Parsed header:", header_parts)

rows = []

# -----------------------------
# Parse each line of the table
# -----------------------------
for line in data_lines:
    parts = [p.strip() for p in line.split("|")]

    # If too many columns → merge extras into the last one
    if len(parts) > len(expected_cols):
        parts = parts[:len(expected_cols)-1] + [" ".join(parts[len(expected_cols)-1:])]

    # If too few columns → pad with empty fields
    if len(parts) < len(expected_cols):
        parts += [""] * (len(expected_cols) - len(parts))

    # If any mismatch remains → skip the line
    if len(parts) != len(expected_cols):
        print("⚠️ Skipping malformed line:", line)
        continue

    # Build a dictionary: {"Day": "..", "Focus": "..", ...}
    row = dict(zip(expected_cols, parts))
    rows.append(row)

# Convert to DataFrame
df_plan = pd.DataFrame(rows)

# Remove completely empty rows (if any slipped in)
df_plan = df_plan.replace("", pd.NA).dropna(how="all").fillna("")

print("\n✅ Parsed workout plan into DataFrame:")
print(df_plan)
print("Shape:", df_plan.shape)

if df_plan.empty:
    raise ValueError("df_plan is empty. The plan text may not have followed the expected table format.")


# -----------------------------------------------------------
# 2) Build the HTML report (header + daily tables)
# -----------------------------------------------------------

# Remove accidental whitespace in column names
df_plan = df_plan.rename(columns=lambda c: c.strip() if isinstance(c, str) else c)

print("\nFinal df_plan columns:", list(df_plan.columns))

# Create a header block summarizing the user's profile
goals_text = ", ".join(user_profile.get("primary_goals", []))

header_html = f"""
<h1>Personalized Workout Plan</h1>
<p><strong>Name:</strong> {user_profile.get("name", "N/A")}</p>
<p><strong>Age:</strong> {user_profile.get("age", "N/A")}</p>
<p><strong>Gender:</strong> {user_profile.get("gender", "N/A")}</p>
<p><strong>Fitness Level:</strong> {user_profile.get("fitness_level", "N/A")}</p>
<p><strong>Goals:</strong> {goals_text}</p>
<p><strong>Days/Week Available:</strong> {user_profile.get("days_per_week_available", "N/A")}</p>
<p><strong>Session Length:</strong> {user_profile.get("session_length_minutes", "N/A")} minutes</p>
"""

if user_profile.get("injuries_or_limitations"):
    header_html += (
        f"<p><strong>Injuries / Limitations:</strong> "
        f"{', '.join(user_profile['injuries_or_limitations'])}</p>"
    )

# Start the HTML output
html_output = [header_html, "<hr>"]

# Columns to show in each day's exercise table
preferred_cols = [
    "Exercise", "Equipment", "Sets", "RepsOrTime", "TargetMuscle", "Intensity"
]

# Safety check: ensure we have a valid Day column
if "Day" not in df_plan.columns:
    df_plan["Day"] = "Plan"

# Loop through each day and build its section
for day_name in df_plan["Day"].unique():
    day_df = df_plan[df_plan["Day"] == day_name].copy()

    focus = day_df["Focus"].iloc[0] if "Focus" in day_df.columns else ""
    warmup = day_df["Warmup"].iloc[0] if "Warmup" in day_df.columns else ""
    cooldown = day_df["Cooldown"].iloc[0] if "Cooldown" in day_df.columns else ""
    notes = day_df["DayNotes"].iloc[0] if "DayNotes" in day_df.columns else ""

    # Keep only the columns we want to show (if they exist)
    display_cols = [c for c in preferred_cols if c in day_df.columns]

    html_output.append(f"<h2>{day_name}</h2>")
    
    if focus:
        html_output.append(f"<p><strong>Focus:</strong> {focus}</p>")
    if warmup:
        html_output.append(f"<p><strong>Warmup:</strong> {warmup}</p>")

    # Insert table of exercises
    if display_cols:
        html_output.append(day_df[display_cols].to_html(index=False, border=1))
    else:
        html_output.append(day_df.to_html(index=False, border=1))

    if cooldown:
        html_output.append(f"<p><strong>Cooldown:</strong> {cooldown}</p>")
    if notes:
        html_output.append(f"<p><strong>Notes:</strong> {notes}</p><br>")

# Join all blocks into a single HTML document
full_html = "\n".join(html_output)

# Save the HTML to disk
output_path = str(OUTPUT_DIR / "workout_plan.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(full_html)

print(f"\n✅ Saved HTML to: {os.path.abspath(output_path)}")
print("Open this file in a web browser, then go to: File → Print → Save as PDF")
from IPython.display import HTML, display
display(HTML(full_html))

In [ ]:
# Cell 8 — Check: does every exercise use equipment we actually have?
detected = " ".join(df_equipment["description"]).lower()
detected_words = {w.rstrip("s") for w in re.findall(r"[a-z]+", detected) if len(w) > 2}
NO_EQUIPMENT = {"none", "bodyweight", "body weight", "n/a", "", "floor", "mat"}

def uses_known_equipment(equipment: str) -> bool:
    e = equipment.strip().lower()
    if e in NO_EQUIPMENT:
        return True
    return any(w.rstrip("s") in detected_words for w in re.findall(r"[a-z]+", e) if len(w) > 2)

df_plan["equipment_ok"] = df_plan["Equipment"].map(uses_known_equipment)
flagged = df_plan.loc[~df_plan["equipment_ok"], ["Day", "Exercise", "Equipment"]]
print(f"{df_plan['equipment_ok'].sum()} of {len(df_plan)} exercises use detected equipment or bodyweight.")
flagged if len(flagged) else print("No exercises need equipment that wasn't detected.")

In [ ]:
# Cell 9 — Save the run, labelled with the models that produced it
run = {"date": date.today().isoformat(), "region": os.environ["WATSONX_URL"],
       "text_model": TEXT_MODEL, "vision_model": VISION_MODEL,
       "equipment": gym_equipment, "exercises": len(df_plan),
       "exercises_using_undetected_equipment": flagged.to_dict("records")}
json.dump(run, open(OUTPUT_DIR / "trainer_run.json", "w"), indent=2)
df_plan.to_csv(OUTPUT_DIR / "workout_plan.csv", index=False)
print("Saved outputs/trainer_run.json, workout_plan.csv and workout_plan.html")